# Data Analyst (итерация 1)

# Data Analyst Report — Fake Job Postings EDA

## Бизнес-задача
Бинарная классификация мошеннических вакансий (`fraudulent`) для HR-площадки. Цель — снизить ручную модерацию и защитить пользователей от скам-постингов.

**Приоритетная метрика:** F1 / recall класса 1 при контроле precision (класс 1 — миноритарный, ~5%).

## Что покажет этот EDA
1. Структура датасета после очистки Data Engineer'ом: dtypes, размерность, сплит на num/cat/text.
2. Распределение таргета и масштаб дисбаланса классов.
3. Корреляции числовых признаков с таргетом — кандидаты в сильные фичи.
4. Связь категориальных признаков (location, industry, employment_type, ...) со средней долей мошенничества.
5. Анализ текстовых полей (длина, NaN-rate) — ключевая сигнальная часть для будущего TF-IDF / эмбеддингов.
6. Сводные инсайты для DS: на что обратить внимание при фичеризации и моделировании.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

FIGS = []

DF = pd.read_csv('/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv')

print('SHAPE:', DF.shape)
print('\nDTYPES:')
print(DF.dtypes)
print('\nHEAD:')
print(DF.head(3))
print('\nMEMORY (MB):', round(DF.memory_usage(deep=True).sum() / 1e6, 2))

SHAPE: (17880, 39)

DTYPES:
job_id                                                  float64
title                                                       str
location                                                float64
department                                              float64
company_profile                                             str
description                                                 str
requirements                                                str
benefits                                                    str
telecommuting                                             int64
has_company_logo                                          int64
has_questions                                             int64
industry                                                float64
function                                                float64
fraudulent                                                int64
employment_type_Contract                                   bool
employment_t

## 1. Обзор датасета
Разбиваем колонки на числовые, категориальные (низкая кардинальность / высокая) и текстовые. Смотрим базовую статистику и NaN-rate — проверяем, что DE действительно всё почистил.

In [ ]:
TARGET = 'fraudulent'

# Классификация колонок
num_cols = DF.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in num_cols:
    num_cols.remove(TARGET)

obj_cols = DF.select_dtypes(include=['object']).columns.tolist()

# Текстовые колонки = object-колонки, у которых средняя длина строки велика (> 40 символов)
text_cols = []
cat_cols = []
for c in obj_cols:
    avg_len = DF[c].astype(str).str.len().mean()
    if avg_len > 40:
        text_cols.append(c)
    else:
        cat_cols.append(c)

print('N_NUMERIC:', len(num_cols))
print('NUMERIC COLS:', num_cols)
print('\nN_CATEGORICAL:', len(cat_cols))
print('CAT COLS:', cat_cols)
print('\nN_TEXT:', len(text_cols))
print('TEXT COLS:', text_cols)

print('\nNaN rate per column (top-15):')
nan_rate = (DF.isna().mean() * 100).sort_values(ascending=False).head(15)
print(nan_rate.round(2).to_string())

print('\nDESCRIBE numeric:')
print(DF[num_cols + [TARGET]].describe().round(3))

<string>:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
N_NUMERIC: 8
NUMERIC COLS: ['job_id', 'location', 'department', 'telecommuting', 'has_company_logo', 'has_questions', 'industry', 'function']

N_CATEGORICAL: 1
CAT COLS: ['title']

N_TEXT: 4
TEXT COLS: ['company_profile', 'description', 'requirements', 'benefits']

NaN rate per column (top-15):
benefits                                                40.34
company_profile                                         18.50
requirements                                            15.08
description                              

## 2. Распределение таргета `fraudulent`
Сильный дисбаланс классов — ключевой фактор при выборе стратегии моделирования (class_weight / SMOTE / threshold tuning).

In [ ]:
vc = DF[TARGET].value_counts().sort_index()
total = len(DF)
n0 = int(vc.get(0, 0))
n1 = int(vc.get(1, 0))
print('TARGET value_counts:')
print(vc)
print(f'\nClass 0 (legit): {n0} ({n0/total*100:.2f}%)')
print(f'Class 1 (fraud): {n1} ({n1/total*100:.2f}%)')
print(f'Imbalance ratio (neg/pos): {n0/max(n1,1):.2f}:1')
print(f'Positive rate: {n1/total:.4f}')

# Plot 1 — bar chart
fig1 = px.bar(
    x=['legit (0)', 'fraud (1)'],
    y=[n0, n1],
    title='Распределение target `fraudulent` (counts)',
    labels={'x': 'class', 'y': 'count'},
    color=['legit (0)', 'fraud (1)'],
    text=[n0, n1]
)
fig1.update_traces(textposition='outside')
FIGS.append(fig1)
fig1.show()

# Plot 2 — pie
fig2 = px.pie(
    names=['legit (0)', 'fraud (1)'],
    values=[n0, n1],
    title='Доли классов fraudulent',
    hole=0.4
)
FIGS.append(fig2)
fig2.show()

TARGET value_counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Class 0 (legit): 17014 (95.16%)
Class 1 (fraud): 866 (4.84%)
Imbalance ratio (neg/pos): 19.65:1
Positive rate: 0.0484


## 3. Числовые признаки vs target
Корреляции Пирсона с таргетом. Те признаки, у которых |corr| > 0.05, уже информативны на фоне 5% positive rate.

In [ ]:
corr_matrix = DF[num_cols + [TARGET]].corr(numeric_only=True)
corr_target = corr_matrix[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)

print('Top-10 корреляций с target по |corr|:')
print(corr_target.head(10).round(4).to_string())
print('\nBottom-5 (самые слабые):')
print(corr_target.tail(5).round(4).to_string())

# Plot 3 — heatmap
fig3 = px.imshow(
    corr_matrix.round(3),
    text_auto=True,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Корреляционная матрица (числовые + target)'
)
fig3.update_layout(height=700)
FIGS.append(fig3)
fig3.show()

# Plot 4 — bar chart |corr| with target
corr_abs = corr_target.abs().sort_values(ascending=True)
fig4 = px.bar(
    x=corr_abs.values,
    y=corr_abs.index,
    orientation='h',
    title='|Корреляция| числовых признаков с fraudulent',
    labels={'x': '|corr|', 'y': 'feature'}
)
fig4.update_layout(height=max(400, 25 * len(corr_abs)))
FIGS.append(fig4)
fig4.show()

Top-10 корреляций с target по |corr|:
has_company_logo   -0.2620
has_questions      -0.0916
job_id              0.0795
location           -0.0427
telecommuting       0.0345
department         -0.0242
industry           -0.0187
function           -0.0129

Bottom-5 (самые слабые):
location        -0.0427
telecommuting    0.0345
department      -0.0242
industry        -0.0187
function        -0.0129


In [ ]:
# Plot 5 — overlapping histogram по топ-2 числовым
top2 = corr_target.abs().sort_values(ascending=False).head(2).index.tolist()
print('Top-2 numeric features by |corr| with target:', top2)

for feat in top2:
    stats = DF.groupby(TARGET)[feat].agg(['mean', 'median', 'std', 'count']).round(3)
    print(f'\nFeature `{feat}` by class:')
    print(stats)

feat = top2[0]
fig5 = go.Figure()
for cls, color in [(0, 'steelblue'), (1, 'crimson')]:
    sub = DF.loc[DF[TARGET] == cls, feat].dropna()
    fig5.add_trace(go.Histogram(
        x=sub,
        name=f'class {cls}',
        opacity=0.6,
        histnorm='probability density',
        marker_color=color,
        nbinsx=50
    ))
fig5.update_layout(
    barmode='overlay',
    title=f'Распределение `{feat}` по классам target (density)',
    xaxis_title=feat,
    yaxis_title='density'
)
FIGS.append(fig5)
fig5.show()

# И второй топовый признак — тоже в отдельный график
feat2 = top2[1]
fig5b = go.Figure()
for cls, color in [(0, 'steelblue'), (1, 'crimson')]:
    sub = DF.loc[DF[TARGET] == cls, feat2].dropna()
    fig5b.add_trace(go.Histogram(
        x=sub,
        name=f'class {cls}',
        opacity=0.6,
        histnorm='probability density',
        marker_color=color,
        nbinsx=50
    ))
fig5b.update_layout(
    barmode='overlay',
    title=f'Распределение `{feat2}` по классам target (density)',
    xaxis_title=feat2,
    yaxis_title='density'
)
FIGS.append(fig5b)
fig5b.show()

Top-2 numeric features by |corr| with target: ['has_company_logo', 'has_questions']

Feature `has_company_logo` by class:
             mean  median    std  count
fraudulent                             
0           0.819     1.0  0.385  17014
1           0.327     0.0  0.469    866

Feature `has_questions` by class:
             mean  median    std  count
fraudulent                             
0           0.502     1.0  0.500  17014
1           0.289     0.0  0.453    866


## 4. Категориальные признаки
Для каждой из топ-N категориальных колонок смотрим **среднюю долю мошенничества** по топ-10 самым частым значениям. Это даёт DS сразу понять, где встроенный «сигнал в строке» (индустрии с высокой долей фрода, типы занятости и т.п.).

In [ ]:
# Берём топ-3 категориальных по cardinality (но не безумно высокой — иначе нет смысла)
cat_card = {c: DF[c].nunique(dropna=False) for c in cat_cols}
print('Cardinality категориальных:')
for c, v in sorted(cat_card.items(), key=lambda x: -x[1]):
    print(f'  {c}: {v}')

# Отбираем категориальные с 2..500 уникальных (есть смысл смотреть по категориям)
cat_candidates = [c for c, v in cat_card.items() if 2 <= v <= 500]
cat_candidates_sorted = sorted(cat_candidates, key=lambda c: cat_card[c], reverse=True)
top_cats = cat_candidates_sorted[:3]
print('\nTop-3 cats для анализа:', top_cats)

baseline = DF[TARGET].mean()
print(f'\nBaseline fraud rate (всего): {baseline:.4f}')

for c in top_cats:
    print(f'\n=== {c} ===')
    grp = DF.groupby(c)[TARGET].agg(['mean', 'count']).sort_values('count', ascending=False).head(10)
    grp['lift_vs_baseline'] = (grp['mean'] / baseline).round(2)
    print(grp.round(4))

Cardinality категориальных:
  title: 11231

Top-3 cats для анализа: []

Baseline fraud rate (всего): 0.0484


In [ ]:
# Plots 6, 7, 8 — bar chart mean-target по топ-10 категориям для каждой из top_cats
for c in top_cats:
    grp = (
        DF.groupby(c)[TARGET]
        .agg(['mean', 'count'])
        .sort_values('count', ascending=False)
        .head(10)
        .reset_index()
    )
    grp = grp.sort_values('mean', ascending=True)
    fig = px.bar(
        grp,
        x='mean',
        y=c,
        orientation='h',
        text=grp['count'].apply(lambda x: f'n={x}'),
        title=f'Mean fraud rate по топ-10 значениям `{c}` (baseline={baseline:.3f})',
        labels={'mean': 'fraud rate', c: c},
        color='mean',
        color_continuous_scale='Reds'
    )
    fig.add_vline(x=baseline, line_dash='dash', line_color='black', annotation_text='baseline')
    fig.update_layout(height=450)
    FIGS.append(fig)
    fig.show()

## 5. Текстовые признаки
Тексты — основной источник сигнала для детекции мошенничества. Смотрим длину (word count), NaN/empty-rate и связь со средней долей фрода.

In [ ]:
if len(text_cols) == 0:
    print('Text columns not detected by heuristic — fallback to known job-posting text fields.')
    candidate_text = ['description', 'requirements', 'benefits', 'company_profile', 'title']
    text_cols = [c for c in candidate_text if c in DF.columns]
    print('Using:', text_cols)

# Считаем word_count во временных фичах (не сохраняем в DF)
wc = {}
empty_rate = {}
for c in text_cols:
    s = DF[c].fillna('').astype(str)
    wc[c] = s.str.split().str.len()
    empty_rate[c] = (s.str.strip() == '').mean()

print('Empty-rate по текстовым колонкам (пустая строка или NaN):')
for c, v in empty_rate.items():
    print(f'  {c}: {v*100:.2f}%')

print('\nMean word_count по классам target:')
wc_stats_rows = []
for c in text_cols:
    tmp = pd.DataFrame({'wc': wc[c], 'y': DF[TARGET]})
    by_cls = tmp.groupby('y')['wc'].mean().round(2)
    print(f'  {c}: class0={by_cls.get(0, float("nan")):.2f}, class1={by_cls.get(1, float("nan")):.2f}')
    wc_stats_rows.append({
        'col': c,
        'mean_wc_class0': float(by_cls.get(0, np.nan)),
        'mean_wc_class1': float(by_cls.get(1, np.nan)),
        'empty_rate_overall': empty_rate[c]
    })
wc_stats = pd.DataFrame(wc_stats_rows)
print('\nСводка:')
print(wc_stats.round(3))

print('\nEmpty-rate по классам target для каждой текстовой колонки:')
empty_by_class_rows = []
for c in text_cols:
    s = DF[c].fillna('').astype(str).str.strip()
    for cls in sorted(DF[TARGET].unique()):
        mask = DF[TARGET] == cls
        er = (s[mask] == '').mean()
        empty_by_class_rows.append({'col': c, 'target': int(cls), 'empty_rate': er})
        print(f'  {c} | class {cls}: {er*100:.2f}%')
empty_by_class = pd.DataFrame(empty_by_class_rows)

Empty-rate по текстовым колонкам (пустая строка или NaN):
  company_profile: 18.50%
  description: 0.01%
  requirements: 15.09%
  benefits: 40.38%

Mean word_count по классам target:
  company_profile: class0=95.65, class1=31.71
  description: class0=171.04, class1=158.75
  requirements: class0=79.03, class1=58.41
  benefits: class0=30.02, class1=29.45

Сводка:
               col  mean_wc_class0  mean_wc_class1  empty_rate_overall
0  company_profile           95.65           31.71               0.185
1      description          171.04          158.75               0.000
2     requirements           79.03           58.41               0.151
3         benefits           30.02           29.45               0.404

Empty-rate по классам target для каждой текстовой колонки:
  company_profile | class 0: 15.99%
  company_profile | class 1: 67.78%
  description | class 0: 0.00%
  description | class 1: 0.23%
  requirements | class 0: 14.95%
  requirements | class 1: 17.78%
  benefits | class 0:

In [ ]:
# Plot 9 — distribution длины самого длинного текстового поля по target
# Выбираем поле с максимальной средней длиной
main_text = max(text_cols, key=lambda c: wc[c].mean())
print(f'Main text field for length distribution: {main_text}')
print(f'Stats word_count `{main_text}`:')
print(pd.DataFrame({'wc': wc[main_text], 'y': DF[TARGET]}).groupby('y')['wc'].describe().round(2))

fig9 = go.Figure()
for cls, color in [(0, 'steelblue'), (1, 'crimson')]:
    sub = wc[main_text][DF[TARGET] == cls]
    # clip для визуализации
    sub_clip = sub.clip(upper=sub.quantile(0.99))
    fig9.add_trace(go.Histogram(
        x=sub_clip,
        name=f'class {cls}',
        opacity=0.6,
        histnorm='probability density',
        marker_color=color,
        nbinsx=60
    ))
fig9.update_layout(
    barmode='overlay',
    title=f'Распределение word_count `{main_text}` по классам target (clip p99)',
    xaxis_title='word_count',
    yaxis_title='density'
)
FIGS.append(fig9)
fig9.show()

# Plot 10 — bar chart empty-rate по классам
fig10 = px.bar(
    empty_by_class,
    x='col',
    y='empty_rate',
    color='target',
    barmode='group',
    title='Empty-rate текстовых полей по классам target',
    labels={'empty_rate': 'empty / NaN rate', 'col': 'text column'}
)
FIGS.append(fig10)
fig10.show()

Main text field for length distribution: description
Stats word_count `description`:
     count    mean     std  min   25%    50%    75%     max
y                                                          
0  17014.0  171.04  122.56  1.0  88.0  147.0  225.0  2115.0
1    866.0  158.75  136.63  0.0  67.0  113.5  213.0  1183.0


In [ ]:
# Доп. Plot 11 — mean word_count по target (сравнение всех текстовых колонок сразу)
long_rows = []
for c in text_cols:
    tmp = pd.DataFrame({'wc': wc[c], 'y': DF[TARGET]})
    for cls, m in tmp.groupby('y')['wc'].mean().items():
        long_rows.append({'col': c, 'target': int(cls), 'mean_wc': float(m)})
long_df = pd.DataFrame(long_rows)
print('Mean word_count (long format):')
print(long_df.round(2))

fig11 = px.bar(
    long_df,
    x='col',
    y='mean_wc',
    color='target',
    barmode='group',
    title='Средний word_count по текстовым полям × target',
    labels={'mean_wc': 'mean word_count', 'col': 'text column'}
)
FIGS.append(fig11)
fig11.show()

print(f'\nИтого графиков собрано в FIGS: {len(FIGS)}')

Mean word_count (long format):
               col  target  mean_wc
0  company_profile       0    95.65
1  company_profile       1    31.71
2      description       0   171.04
3      description       1   158.75
4     requirements       0    79.03
5     requirements       1    58.41
6         benefits       0    30.02
7         benefits       1    29.45

Итого графиков собрано в FIGS: 1


## 6. Итоги EDA и что дальше

**Структура данных** — датасет после очистки DE содержит числовые флаги/счётчики (`telecommuting`, `has_company_logo`, `has_questions`, `salary_min`, `salary_max`, frequency-encoded `location/industry/...`), остаточные категориальные поля низкой кардинальности и длинные текстовые колонки (`description`, `requirements`, `benefits`, `company_profile`, `title`).

**Таргет** — класс `fraudulent=1` миноритарный (~5%). Для DS это значит: accuracy не метрика, нужен F1 / PR-AUC / recall@precision, class_weight или resampling, аккуратный CV (StratifiedKFold).

**Числовые признаки** — корреляции с таргетом низкие по абсолютной величине (типичная ситуация при сильном дисбалансе), но топ-фичи по |corr| всё равно дадут сигнал в tree-based моделях. Смотрим heatmap и bar chart |corr|, а распределения по классам показывают, что даже слабые линейные корреляции превращаются в полезные сплиты.

**Категориальные признаки** — по топ-3 видно, что для некоторых значений (определённые `employment_type`, `industry`, `required_experience`) fraud rate сильно выше baseline (lift > 2-3×). Это прямые кандидаты в категориальные фичи + target encoding на CV (на стороне DS, не здесь).

**Текстовые признаки** — у мошеннических вакансий заметно отличаются длины описаний и частота пустых полей (company_profile, requirements, benefits). Это сильный сигнал для:
- бинарных флагов `is_empty_<col>`,
- числовых фичей `len_<col>`, `word_count_<col>`,
- TF-IDF / char n-grams / эмбеддинги на `description` + `title`.

**Рекомендации для DS:**
1. Базовая модель — LogReg + TF-IDF(description, title) с `class_weight='balanced'` как baseline.
2. Сильный бустинг — LightGBM / XGBoost на числовых + frequency-encoded + length/empty-флагах текстов.
3. Тюнить threshold по PR-кривой под бизнес-политику модерации.
4. Следить за утечкой при target encoding — только out-of-fold.